
# 🥈 NB02 — Bronze → Silver: ETL & Data Preparation
## Investor Intelligence Platform — FIIs Brasil 🇧🇷
### CRISP-DM Phase: Data Preparation

| | |
|---|---|
| **Input** | `data/external/*.parquet` — Bronze frozen (NB01) |
| **Output** | `data/silver/silver_articles.parquet` |
| **Schema** | 22 colunas (17 Bronze − `raw_html` + 7 Silver-new) |
| **Engine** | PySpark DataFrame API + RDD validation |

## 🎯 O que este notebook faz

1. Carrega de `data/external/` (nunca de fontes ao vivo).
2. Limpa texto: HTML → entidades → URLs → boilerplate.
3. Normaliza datas para ISO 8601 UTC.
4. Mapeia domínios para labels legíveis.
5. Computa `word_count`, `char_count`, `has_content`.
6. Valida via pipeline RDD.
7. Aplica 3 quality gates.
8. Grava Silver Parquet + relatório de processamento.

## 📦 Seção 1 — Imports

In [6]:

import sys, os
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import re, json, random, warnings, unicodedata
from datetime import datetime, timezone
from html import unescape
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
from dateutil import parser as dateutil_parser

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, BooleanType
from pyspark.sql.window import Window
import pyspark

warnings.filterwarnings("ignore")
print(f"Python  : {sys.version.split()[0]}")
print(f"PySpark : {pyspark.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"PyArrow : {pa.__version__}")

Python  : 3.12.12
PySpark : 4.1.2
Pandas  : 3.0.3
PyArrow : 24.0.0


## ⚙️ Seção 2 — Configuração e SparkSession

In [7]:
PROJECT_ROOT = Path(os.getcwd()).resolve()
_lim = 6
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT and _lim > 0:
    PROJECT_ROOT = PROJECT_ROOT.parent; _lim -= 1
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import (
    EXTERNAL_DIR, BRONZE_DIR, RANDOM_SEED,
    SPARK_DRIVER_MEMORY, SPARK_APP_NAME, SPARK_SHUFFLE_PARTS,
)
from src.utils.logger import ingestion_logger, log_quality_check, log_spark_start

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

EXTERNAL_DIR = Path(EXTERNAL_DIR)
SILVER_DIR   = PROJECT_ROOT / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

MIN_WORD_COUNT = 30
MIN_CHAR_COUNT = 150

spark = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}_silver")
    .master("local[*]")
    .config("spark.driver.memory",          SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTS)
    .config("spark.sql.adaptive.enabled",   "false")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

logger = ingestion_logger()
log_spark_start(logger, spark.sparkContext.appName, spark.version)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"EXTERNAL_DIR : {EXTERNAL_DIR}")
print(f"SILVER_DIR   : {SILVER_DIR}")
print(f"SparkSession : {spark.sparkContext.appName} | Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 15:44:49 WARN Utils: Your hostname, MacBook-Air-100.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.4 instead (on interface en0)
26/06/14 15:44:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:44:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-06-14T15:44:52 | INFO     | fii_pipeline.ingestion | SPARK_START | app=FIIIntelligencePlatform_silver | spark=4.1.2
PROJECT_ROOT : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform
EXTERNAL_DIR : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/external
SILVER_DIR   : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/silver
SparkSession : FIIIntelligencePlatform_silver | Spark 4.1.2



## 📂 Seção 3 — Carregamento do Bronze Frozen

**Regra crítica:** NB02 lê **apenas** de `data/external/`. Nunca chama APIs ao vivo.

**Estratégia resiliente (3 tentativas):**
```
1. bronze_all_articles.parquet (combined)
2. rss_*.parquet + portal_*.parquet + reddit_*.parquet (partes)
3. glob *.parquet em data/external/
```

In [8]:

_COMBINED = [
    EXTERNAL_DIR / "bronze_all_articles.parquet",
    EXTERNAL_DIR / "fii_articles_bronze.parquet",
]
_PARTS = [
    "rss_articles.parquet",      "rss_fii_articles.parquet",
    "portal_articles.parquet",   "portal_fii_articles.parquet",
    "reddit_posts.parquet",      "reddit_fii_posts.parquet",
]

df_bronze = None

for _cp in _COMBINED:
    if _cp.exists():
        df_bronze = pd.read_parquet(_cp)
        print(f"  Combined: {_cp.name} → {len(df_bronze):,} registros")
        break

if df_bronze is None:
    _parts, _seen = [], set()
    for _fname in _PARTS:
        _p = EXTERNAL_DIR / _fname
        if _p.exists() and _fname not in _seen:
            _df = pd.read_parquet(_p)
            _parts.append(_df)
            _seen.add(_fname)
            print(f"  {_fname}: {len(_df):,} registros")
    if _parts:
        df_bronze = pd.concat(_parts, ignore_index=True)

if df_bronze is None:
    _globs = list(EXTERNAL_DIR.glob("*.parquet"))
    if _globs:
        df_bronze = pd.concat([pd.read_parquet(p) for p in _globs], ignore_index=True)
        print(f"  Glob: {len(_globs)} arquivo(s) → {len(df_bronze):,} registros")

if df_bronze is None or df_bronze.empty:
    raise FileNotFoundError(
        f"Bronze não encontrado em {EXTERNAL_DIR}. Execute NB01 primeiro."
    )

print(f"\n✅ Bronze carregado: {len(df_bronze):,} registros")
print(f"   Colunas: {list(df_bronze.columns)}")
print(f"\n   source_type:\n{df_bronze['source_type'].value_counts().to_string()}")

  Combined: bronze_all_articles.parquet → 503 registros

✅ Bronze carregado: 503 registros
   Colunas: ['article_id', 'source', 'source_type', 'title', 'content', 'summary', 'url', 'author', 'published_at', 'collected_at', 'language', 'tags', 'query_used', 'ingestion_method', 'raw_html', 'content_hash', 'metadata_json']

   source_type:
source_type
reddit      345
scraping     89
rss          69


## 🧹 Seção 4 — UDFs de Limpeza e Normalização

In [9]:

# ── Limpeza de texto ──────────────────────────────────────────────────────────
_URL_RE  = re.compile(r'https?://\S+|www\.\S+')
_TAG_RE  = re.compile(r'<[^>]+>')
_WS_RE   = re.compile(r'\s+')
_BOILER  = ['leia mais', 'veja também', 'acesse aqui', 'clique aqui',
            'saiba mais', 'newsletter', 'assine já', 'você também pode']

def clean_text(raw: str) -> str:
    if not raw or not isinstance(raw, str): return ""
    t = unescape(raw)
    t = _TAG_RE.sub(' ', t)
    t = _URL_RE.sub(' ', t)
    for b in _BOILER:
        t = t.replace(b, ' ')
    t = _WS_RE.sub(' ', t)
    return t.strip()

_clean_text_udf = F.udf(clean_text, StringType())

# ── Normalização de datas ─────────────────────────────────────────────────────
def parse_date(raw_date) -> str:
    if raw_date is None or (isinstance(raw_date, str) and raw_date.strip() == ""):
        return None
    try:
        dt = dateutil_parser.parse(str(raw_date), fuzzy=True)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    except Exception:
        return None

_parse_date_udf = F.udf(parse_date, StringType())

# ── Labels de fontes ──────────────────────────────────────────────────────────
SOURCE_LABELS = {
    "infomoney.com.br":               "InfoMoney",
    "empiricus.com.br":               "Empiricus",
    "moneytimes.com.br":              "Money Times",
    "seudinheiro.com":                "Seu Dinheiro",
    "exame.com":                      "Exame Invest",
    "cnnbrasil.com.br":               "CNN Brasil Business",
    "sunoresearch.com.br":            "Suno Research",
    "einvestidor.estadao.com.br":     "E-Investidor Estadão",
    "neofeed.com.br":                 "NeoFeed",
    "blog.toroinvestimentos.com.br":  "Toro Investimentos",
    "fundsexplorer.com.br":           "Funds Explorer",
    "statusinvest.com.br":            "Status Invest",
    "clubefii.com.br":                "Clube FII",
    "fiis.com.br":                    "FIIs.com.br",
    "portaldofii.com.br":             "Portal do FII",
    "investidor10.com.br":            "Investidor10",
    "euqueroinvestir.com":            "Eu Quero Investir",
    "borainvestir.b3.com.br":         "Bora Investir (B3)",
    "conteudos.xpi.com.br":           "XP Conteúdos",
    "br.investing.com":               "Investing Brasil",
    "reddit.com":                     "Reddit (r/investimentos · r/farialimabets)",
}

def source_label(source: str) -> str:
    if not source: return "Desconhecido"
    return SOURCE_LABELS.get(source, source)

_source_label_udf = F.udf(source_label, StringType())

print("✅ UDFs definidas: clean_text | parse_date | source_label")

✅ UDFs definidas: clean_text | parse_date | source_label


## ⚡ Seção 5 — Pipeline Spark: Bronze → Silver

In [10]:

# ── Criar Spark DataFrame a partir do Bronze pandas ───────────────────────────
sdf_bronze = spark.createDataFrame(df_bronze.fillna(""))

# ── Aplicar transformações via UDFs ──────────────────────────────────────────
sdf_silver = (
    sdf_bronze
    .withColumn("text_clean",    _clean_text_udf(
        F.coalesce(F.col("content"), F.col("summary"), F.col("title"))
    ))
    .withColumn("published_dt",  _parse_date_udf(F.col("published_at")))
    .withColumn("source_label",  _source_label_udf(F.col("source")))
    .withColumn("word_count",    F.size(F.split(F.col("text_clean"), "\\s+")))
    .withColumn("char_count",    F.length(F.col("text_clean")))
    .withColumn("has_content",   F.col("word_count") >= MIN_WORD_COUNT)
)

print("✅ Transformações aplicadas: text_clean | published_dt | source_label | word_count | char_count | has_content")
print(f"   Registros: {sdf_silver.count():,}")

✅ Transformações aplicadas: text_clean | published_dt | source_label | word_count | char_count | has_content


   Registros: 503


## 🚦 Seção 6 — Quality Gates (3 Barreiras)

In [11]:

n_total = sdf_silver.count()

# ── Gate 1: article_id e url não nulos ────────────────────────────────────────
sdf_g1    = sdf_silver.filter(
    F.col("article_id").isNotNull() & (F.col("article_id") != "") &
    F.col("url").isNotNull() & (F.col("url") != "")
)
n_g1 = sdf_g1.count()
log_quality_check(logger, gate="G1_null_ids", passed=n_g1, dropped=n_total - n_g1)
print(f"Gate 1 — IDs não nulos:  {n_g1:,}  (descartados: {n_total - n_g1:,})")

# ── Gate 2: word_count mínimo ─────────────────────────────────────────────────
sdf_g2    = sdf_g1.filter(F.col("word_count") >= MIN_WORD_COUNT)
n_g2 = sdf_g2.count()
log_quality_check(logger, gate="G2_word_count", passed=n_g2, min_words=MIN_WORD_COUNT)
print(f"Gate 2 — word_count≥{MIN_WORD_COUNT}: {n_g2:,}  (descartados: {n_g1 - n_g2:,})")

# ── Gate 3: deduplicação por article_id via Window ─────────────────────────────
_w       = Window.partitionBy("article_id").orderBy(F.col("collected_at").desc())
sdf_g3   = (
    sdf_g2
    .withColumn("_rn", F.row_number().over(_w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
n_g3 = sdf_g3.count()
log_quality_check(logger, gate="G3_dedup", passed=n_g3, dropped=n_g2 - n_g3)
print(f"Gate 3 — dedup:          {n_g3:,}  (removidos: {n_g2 - n_g3:,})")

# ── Validação RDD: verificar integridade dos article_id ───────────────────────
_ids_rdd = sdf_g3.select("article_id").rdd.map(lambda r: r["article_id"])
_id_lens = _ids_rdd.map(lambda aid: len(aid)).distinct().collect()
assert all(l == 64 for l in _id_lens), f"article_id deve ter 64 chars (SHA-256), encontrado: {_id_lens}"
print(f"\n✅ RDD validation: todos os article_id têm 64 chars (SHA-256)")
print(f"   Total Silver após quality gates: {n_g3:,} registros")

2026-06-14T15:44:56 | INFO     | fii_pipeline.ingestion | QUALITY_CHECK | gate=G1_null_ids | passed=503 | dropped=0
Gate 1 — IDs não nulos:  503  (descartados: 0)


2026-06-14T15:44:58 | INFO     | fii_pipeline.ingestion | QUALITY_CHECK | gate=G2_word_count | passed=126 | min_words=30


Gate 2 — word_count≥30: 126  (descartados: 377)


2026-06-14T15:44:59 | INFO     | fii_pipeline.ingestion | QUALITY_CHECK | gate=G3_dedup | passed=126 | dropped=0


Gate 3 — dedup:          126  (removidos: 0)



✅ RDD validation: todos os article_id têm 64 chars (SHA-256)
   Total Silver após quality gates: 126 registros


## 💾 Seção 7 — Salvar Silver Parquet

In [12]:

# ── Selecionar colunas Silver finais ──────────────────────────────────────────
SILVER_COLS = [
    "article_id", "source", "source_type", "source_label",
    "title", "url", "author", "published_at", "published_dt", "collected_at",
    "language", "tags", "query_used", "ingestion_method",
    "text_clean", "word_count", "char_count", "has_content",
    "content_hash", "metadata_json",
]

_available = [c for c in SILVER_COLS if c in sdf_g3.columns]
sdf_final  = sdf_g3.select(*_available)

# ── Converter para pandas e salvar ───────────────────────────────────────────
df_silver = sdf_final.toPandas()
silver_path = SILVER_DIR / "silver_articles.parquet"
df_silver.to_parquet(silver_path, index=False, compression="snappy")

print(f"✅ Silver gravado: {silver_path}")
print(f"   {len(df_silver):,} registros × {len(df_silver.columns)} colunas")
print(f"\n   Distribuição source_type:")
print(df_silver['source_type'].value_counts().to_string())
print(f"\n   word_count — média: {df_silver['word_count'].mean():.1f}  | total: {df_silver['word_count'].sum():,}")
print(f"   has_content=True : {df_silver['has_content'].sum():,}")
print(f"\n   Top 5 fontes:")
print(df_silver['source_label'].value_counts().head().to_string())

✅ Silver gravado: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/silver/silver_articles.parquet
   126 registros × 20 colunas

   Distribuição source_type:
source_type
rss         69
scraping    55
reddit       2

   word_count — média: 593.5  | total: 74,786
   has_content=True : 126

   Top 5 fontes:
source_label
Empiricus              32
CNN Brasil Business    17
Funds Explorer         12
Investidor10           10
FIIs.com.br            10


## 📊 Seção 8 — Relatório de Processamento

In [13]:

report = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "bronze_records_loaded": n_total,
    "gate_1_null_ids":       n_g1,
    "gate_2_word_count":     n_g2,
    "gate_3_dedup":          n_g3,
    "silver_records_final":  len(df_silver),
    "has_content_true":      int(df_silver['has_content'].sum()),
    "source_type_counts":    df_silver['source_type'].value_counts().to_dict(),
    "min_word_count_threshold": MIN_WORD_COUNT,
    "spark_version": spark.version,
    "random_seed": RANDOM_SEED,
}

report_path = SILVER_DIR / "silver_processing_report.json"
import json as _json
report_path.write_text(_json.dumps(report, indent=2, ensure_ascii=False))

print("✅ Relatório de processamento:")
for k, v in report.items():
    print(f"   {k}: {v}")

✅ Relatório de processamento:
   timestamp: 2026-06-14T18:45:02.795522+00:00
   bronze_records_loaded: 503
   gate_1_null_ids: 503
   gate_2_word_count: 126
   gate_3_dedup: 126
   silver_records_final: 126
   has_content_true: 126
   source_type_counts: {'rss': 69, 'scraping': 55, 'reddit': 2}
   min_word_count_threshold: 30
   spark_version: 4.1.2
   random_seed: 42



## ✅ NB02 Completo

| Artefato | Localização |
|----------|-------------|
| `silver_articles.parquet` | `data/silver/` |
| `silver_processing_report.json` | `data/silver/` |

▶️ **Próximo:** `03_word_count_mapreduce.ipynb`